# 安装依赖


In [1]:
import os 
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

!pip install -q langchain-text-splitters langchain-openai langchain-community langchain-huggingface faiss-cpu sentence-transformers rank-bm25
print("All packages installed successfully!")

All packages installed successfully!


## 导入模块

In [4]:
import os
import re
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
print("All imports successful!")

All imports successful!


## 配置Deepseek llm

In [10]:
DEEPSEEK_API_KEY = "sk-c65beac8fcff424a905a9c7325f70f2c"
DEEPSEEK_API_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL= "deepseek-chat"

llm = ChatOpenAI(
    base_url=DEEPSEEK_API_URL,
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
    temperature=0.7,
    max_tokens=2048,
    timeout=60,
)

# 快速测试链接
try:
    response = llm.invoke("Hello, world!")
    print("LLM response:", response)
except Exception as e:
    print("Error occurred:", e)

LLM response: content='Hello! How can I help you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '3a6dc122-ae8e-42ce-8cd2-d592168d0459', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ebae9-9985-7072-998a-99addd6d61c4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


## 加载本地知识库

In [ ]:
def clean_document(text: str) -> str:
    text=re.sub(r'!\[Image\]\(https://internal-api-drive-stream\.feishu\.cn[^)]+\)','',text)#去除飞书到处的图片链接残留
    text=re.sub(r'!\[Image\]\(data:image[^)]+\)','',text)#去除其他图片链接残留
    text=re.sub(r'\n{3,}','\n\n',text)
    text=re.sub(r' +$','',text)
    text=text.replace('\\-','-').replace('\\_','_')
    return text.strip()

KNOWLEDGE_DIR = './knowledge_base'
Path(KNOWLEDGE_DIR).mkdir(parents=True, exist_ok=True)